# PSX AI — model training (Colab Pro)

Trains XGBoost + Random Forest + LSTM + a logistic-stacking meta-model
per symbol, exports each to ONNX, saves to Google Drive.

Prerequisites (see `README.md` next to this notebook):
1. `scripts/export_training_data.py` was run locally
2. The resulting `psx_features.parquet` was uploaded to your Google
   Drive at `MyDrive/psx-ai/training/psx_features.parquet`
3. You opened this notebook in Colab and set Runtime → GPU (T4 or V100)

After everything runs (~3–6 hours unattended), download the
`models/onnx/` folder from Drive and commit it under
`psx-inference/models/onnx/` in the repo. Inference from then on
is CPU-only via `onnxruntime`.

## 0. Sanity check the runtime

In [ ]:
import torch, sys, platform
print('Python:', sys.version.split()[0], '|', platform.system(), platform.release())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')

## 1. Install training-only dependencies

Most are pre-installed on Colab; `skl2onnx` + `onnxmltools` + `onnxruntime`
are not. ~30s on a fresh runtime.

In [ ]:
!pip install -q 'xgboost==2.1.3' 'scikit-learn==1.5.2' 'skl2onnx==1.17.0' \
                'onnxmltools==1.12.0' 'onnxruntime==1.20.1' 'onnx==1.16.0' \
                'pandas==2.2.3' 'pyarrow' 'tqdm'

## 2. Mount Google Drive

Colab will pop up a consent screen the first time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE_BASE = pathlib.Path('/content/drive/MyDrive/psx-ai')
TRAINING_PARQUET = DRIVE_BASE / 'training' / 'psx_features.parquet'
MODEL_OUTPUT_DIR = DRIVE_BASE / 'models' / 'onnx'
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAINING_PARQUET.exists(), (
    f'Could not find {TRAINING_PARQUET}. Upload psx_features.parquet to '
    f'MyDrive/psx-ai/training/ first.'
)
print('Will write models to:', MODEL_OUTPUT_DIR)

## 3. Load training data

Walk-forward time-series split: train on the first 70%, validate on the
next 15%, test on the last 15%. **No random shuffle** — we never train
on data that's chronologically after the test set.

In [ ]:
import pandas as pd, numpy as np

FEATURE_NAMES = (
    'log_return_1d', 'log_return_5d', 'log_return_10d', 'log_return_20d',
    'momentum_10', 'disparity_5', 'disparity_10', 'rsi_14',
    'stochastic_k', 'stochastic_d', 'williams_r_14', 'macd', 'macd_signal',
    'macd_histogram', 'bollinger_pct_b', 'atr_14_norm', 'obv_change_5d',
    'volume_z_20', 'price_z_50', 'kse100_return_pct', 'sector_return_pct',
    'kibor_6m_pct', 'pkr_usd_change_pct',
)

df = pd.read_parquet(TRAINING_PARQUET)
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)
df[list(FEATURE_NAMES)] = df[list(FEATURE_NAMES)].astype(np.float32).fillna(0.0)
df['y_next_day_up'] = df['y_next_day_up'].astype(np.int32)

print(f'Loaded {len(df):,} rows × {df.shape[1]} cols')
print(f'Symbols: {df.symbol.nunique()}')
print(f'Date range: {df.date.min()} → {df.date.max()}')
print(f'Class balance: y=1 {df.y_next_day_up.mean()*100:.1f}%')

## 4. Per-symbol training loop

For each symbol we fit:
1. **XGBoost** (mean-reversion / non-linear interactions) — fast, no GPU
2. **Random Forest** (trend / robust to outliers) — fast, no GPU
3. **LSTM** (sequence-aware, 30-day windows) — GPU-accelerated
4. **Logistic stacking** — combines the three sub-model probabilities

Sub-models that fail to beat 53% AUC on validation are dropped and the
symbol is flagged `predictions_disabled` (the runtime checks this).

**Run time on a T4:** ~3-4 minutes per symbol × 84 symbols = ~5 hours.
Tab away — Colab Pro keeps the kernel alive.

In [ ]:
import json, time, warnings
from dataclasses import dataclass, asdict
from typing import Optional

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MIN_TRAIN_AUC = 0.53          # below this the model is dropped
LSTM_WINDOW = 30              # days of history per LSTM input


@dataclass
class SymbolReport:
    symbol: str
    n_train: int
    n_val: int
    n_test: int
    auc_xgb: Optional[float]
    auc_rf:  Optional[float]
    auc_lstm: Optional[float]
    auc_meta: Optional[float]
    predictions_disabled: bool
    reason: Optional[str]


def _split_walk_forward(g: pd.DataFrame):
    n = len(g)
    a, b = int(n * 0.70), int(n * 0.85)
    return g.iloc[:a], g.iloc[a:b], g.iloc[b:]


def _train_xgb(X_tr, y_tr, X_val, y_val):
    m = xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
        random_state=42, early_stopping_rounds=30,
        tree_method='hist',
    )
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return m, roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])


def _train_rf(X_tr, y_tr, X_val, y_val):
    m = RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=5,
        n_jobs=-1, random_state=42,
    )
    m.fit(X_tr, y_tr)
    return m, roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])


class LSTMClassifier(nn.Module):
    def __init__(self, n_features: int, hidden: int = 32):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.head = nn.Sequential(nn.Linear(hidden, 16), nn.ReLU(), nn.Linear(16, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])


def _windows(X: np.ndarray, y: np.ndarray, w: int):
    if len(X) <= w: return None, None
    xs = np.stack([X[i - w:i] for i in range(w, len(X))]).astype(np.float32)
    ys = y[w:].astype(np.float32)
    return xs, ys


def _train_lstm(X_tr, y_tr, X_val, y_val):
    tr_x, tr_y = _windows(X_tr, y_tr, LSTM_WINDOW)
    val_x, val_y = _windows(X_val, y_val, LSTM_WINDOW)
    if tr_x is None or val_x is None or len(tr_x) < 200: return None, None
    loader = DataLoader(TensorDataset(torch.tensor(tr_x), torch.tensor(tr_y)), batch_size=64, shuffle=True)
    model = LSTMClassifier(X_tr.shape[1]).to(DEVICE)
    opt, loss_fn = optim.Adam(model.parameters(), lr=1e-3), nn.BCEWithLogitsLoss()
    best_auc = -1.0
    best_state = None
    for epoch in range(8):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss_fn(model(xb).squeeze(-1), yb).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            preds = torch.sigmoid(model(torch.tensor(val_x).to(DEVICE)).squeeze(-1)).cpu().numpy()
        auc = roc_auc_score(val_y, preds) if len(set(val_y)) > 1 else 0.5
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state: model.load_state_dict(best_state)
    return model, best_auc


def _meta_probs(sub_probs: dict) -> np.ndarray:
    cols = [v for v in sub_probs.values() if v is not None]
    if not cols: return None
    return np.column_stack(cols)


REPORTS: list[SymbolReport] = []
MODELS: dict[str, dict] = {}        # symbol → {'xgb': ..., 'rf': ..., 'lstm': ..., 'meta': ...}
MIN_ROWS_PER_SYMBOL = 200

In [ ]:
from tqdm.auto import tqdm

symbols = sorted(df.symbol.unique().tolist())
print(f'Training {len(symbols)} symbols on {DEVICE}…')

for sym in tqdm(symbols):
    g = df[df.symbol == sym].copy()
    if len(g) < MIN_ROWS_PER_SYMBOL:
        REPORTS.append(SymbolReport(sym, len(g), 0, 0, None, None, None, None,
                                    True, f'only {len(g)} rows'))
        continue

    tr, va, te = _split_walk_forward(g)
    X_tr, y_tr = tr[list(FEATURE_NAMES)].values, tr['y_next_day_up'].values
    X_va, y_va = va[list(FEATURE_NAMES)].values, va['y_next_day_up'].values
    X_te, y_te = te[list(FEATURE_NAMES)].values, te['y_next_day_up'].values
    if len(set(y_va)) < 2 or len(set(y_te)) < 2:
        REPORTS.append(SymbolReport(sym, len(tr), len(va), len(te), None, None, None, None,
                                    True, 'val/test labels all one class'))
        continue

    try:
        xgb_m, auc_x = _train_xgb(X_tr, y_tr, X_va, y_va)
    except Exception as e:
        xgb_m, auc_x = None, None
        print(f'  xgb {sym}: {e}')

    try:
        rf_m, auc_r = _train_rf(X_tr, y_tr, X_va, y_va)
    except Exception as e:
        rf_m, auc_r = None, None

    try:
        lstm_m, auc_l = _train_lstm(X_tr, y_tr, X_va, y_va)
    except Exception as e:
        lstm_m, auc_l = None, None

    # Drop weak sub-models
    if auc_x is not None and auc_x < MIN_TRAIN_AUC: xgb_m, auc_x = None, None
    if auc_r is not None and auc_r < MIN_TRAIN_AUC: rf_m, auc_r = None, None
    if auc_l is not None and auc_l < MIN_TRAIN_AUC: lstm_m, auc_l = None, None

    if not any([xgb_m, rf_m, lstm_m]):
        REPORTS.append(SymbolReport(sym, len(tr), len(va), len(te), auc_x, auc_r, auc_l, None,
                                    True, 'all sub-models failed validation'))
        continue

    # Stacking meta-model on the validation set's sub-model probabilities.
    sub_val = {}
    if xgb_m  is not None: sub_val['xgb']  = xgb_m.predict_proba(X_va)[:, 1]
    if rf_m   is not None: sub_val['rf']   = rf_m.predict_proba(X_va)[:, 1]
    if lstm_m is not None:
        lstm_m.eval()
        val_x, _ = _windows(X_va, y_va, LSTM_WINDOW)
        with torch.no_grad():
            preds = torch.sigmoid(lstm_m(torch.tensor(val_x).to(DEVICE)).squeeze(-1)).cpu().numpy()
        # Pad to align rows: LSTM is shorter by LSTM_WINDOW; backfill mean for the stacking input.
        full = np.full(len(X_va), preds.mean())
        full[LSTM_WINDOW:] = preds
        sub_val['lstm'] = full

    Xmeta_va = _meta_probs(sub_val)
    meta = LogisticRegression(max_iter=200).fit(Xmeta_va, y_va)
    auc_m = roc_auc_score(y_va, meta.predict_proba(Xmeta_va)[:, 1])

    MODELS[sym] = {'xgb': xgb_m, 'rf': rf_m, 'lstm': lstm_m, 'meta': meta,
                   'sub_keys': list(sub_val.keys())}
    REPORTS.append(SymbolReport(sym, len(tr), len(va), len(te), auc_x, auc_r, auc_l, auc_m, False, None))

report_df = pd.DataFrame([asdict(r) for r in REPORTS])
print(f'\nTrained: {(~report_df.predictions_disabled).sum()} / {len(report_df)} symbols')
print(f'Mean meta-AUC (enabled symbols): {report_df.loc[~report_df.predictions_disabled, "auc_meta"].mean():.3f}')
report_df.head(20)

## 5. Export every trained model to ONNX

ONNX is framework-agnostic — once exported, runtime inference uses
`onnxruntime` only (no torch / no sklearn at serve time). The full set
of files is ~30 MB.

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from onnxmltools.convert import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType as MLToolsFloat
import torch.onnx

N_FEATURES = len(FEATURE_NAMES)
exported = 0

for sym, bundle in tqdm(MODELS.items()):
    sym_dir = MODEL_OUTPUT_DIR / sym
    sym_dir.mkdir(parents=True, exist_ok=True)

    # XGBoost
    if bundle['xgb'] is not None:
        onx = convert_xgboost(bundle['xgb'], initial_types=[('X', MLToolsFloat([None, N_FEATURES]))])
        (sym_dir / 'xgb.onnx').write_bytes(onx.SerializeToString())

    # Random Forest
    if bundle['rf'] is not None:
        onx = convert_sklearn(bundle['rf'], initial_types=[('X', FloatTensorType([None, N_FEATURES]))])
        (sym_dir / 'rf.onnx').write_bytes(onx.SerializeToString())

    # LSTM
    if bundle['lstm'] is not None:
        bundle['lstm'].eval().cpu()
        dummy = torch.zeros(1, LSTM_WINDOW, N_FEATURES)
        torch.onnx.export(
            bundle['lstm'], dummy, str(sym_dir / 'lstm.onnx'),
            input_names=['X'], output_names=['logit'],
            dynamic_axes={'X': {0: 'batch'}, 'logit': {0: 'batch'}},
            opset_version=14,
        )

    # Meta model
    onx = convert_sklearn(
        bundle['meta'],
        initial_types=[('X', FloatTensorType([None, len(bundle['sub_keys'])]))],
    )
    (sym_dir / 'meta.onnx').write_bytes(onx.SerializeToString())

    # Manifest tells the runtime which sub-models exist for this symbol.
    (sym_dir / 'manifest.json').write_text(json.dumps({
        'symbol': sym,
        'sub_keys': bundle['sub_keys'],
        'feature_names': list(FEATURE_NAMES),
        'lstm_window': LSTM_WINDOW,
    }, indent=2))
    exported += 1

print(f'\n✓ Exported {exported} symbols to {MODEL_OUTPUT_DIR}')

## 6. Save the training report

Useful audit trail — which symbols passed, which AUCs each sub-model
scored, which got disabled and why.

In [ ]:
report_path = MODEL_OUTPUT_DIR / 'training_report.csv'
report_df.to_csv(report_path, index=False)
print(f'Report saved: {report_path}')

print('\n--- enabled symbols ---')
print(report_df.loc[~report_df.predictions_disabled, ['symbol', 'auc_xgb', 'auc_rf', 'auc_lstm', 'auc_meta']].to_string(index=False))
print('\n--- disabled symbols ---')
print(report_df.loc[report_df.predictions_disabled, ['symbol', 'reason']].to_string(index=False))

## 7. Sanity-check one model via onnxruntime

Re-load a trained model the way the inference service will, score one
row, eyeball the result.

In [ ]:
import onnxruntime as ort

enabled = report_df.loc[~report_df.predictions_disabled, 'symbol'].tolist()
if enabled:
    test_sym = enabled[0]
    manifest = json.loads((MODEL_OUTPUT_DIR / test_sym / 'manifest.json').read_text())
    print(f'Sanity-checking {test_sym}')
    print(f'  sub-models present: {manifest["sub_keys"]}')
    sub_probs = []
    fake_features = np.zeros((1, N_FEATURES), dtype=np.float32)
    for k in manifest['sub_keys']:
        sess = ort.InferenceSession(str(MODEL_OUTPUT_DIR / test_sym / f'{k}.onnx'))
        inp = sess.get_inputs()[0].name
        if k == 'lstm':
            x = np.zeros((1, manifest['lstm_window'], N_FEATURES), dtype=np.float32)
            out = sess.run(None, {inp: x})
            prob = 1 / (1 + np.exp(-out[0])).squeeze()
        else:
            out = sess.run(None, {inp: fake_features})
            # sklearn / xgb ONNX models return [labels, probabilities]
            prob = out[1][0][1]
        print(f'  {k}: prob_up={prob:.4f}')
        sub_probs.append(prob)
    meta_sess = ort.InferenceSession(str(MODEL_OUTPUT_DIR / test_sym / 'meta.onnx'))
    meta_in = meta_sess.get_inputs()[0].name
    meta_out = meta_sess.run(None, {meta_in: np.array([sub_probs], dtype=np.float32)})
    print(f'  meta: prob_up={meta_out[1][0][1]:.4f}')
else:
    print('No enabled symbols to test — review the training report above.')

## Done

Your `MyDrive/psx-ai/models/onnx/` folder now contains:

```
models/onnx/
├── training_report.csv
├── HBL/
│   ├── xgb.onnx
│   ├── rf.onnx
│   ├── lstm.onnx
│   ├── meta.onnx
│   └── manifest.json
├── UBL/
│   └── ...
└── ...
```

**Next steps (on your laptop):**
1. Download the entire `models/onnx/` folder from Drive (right-click → Download)
2. Drop it into the repo at `psx-inference/models/onnx/`
3. `git add psx-inference/models/onnx/ && git commit -m "feat: trained ML models v1"`
4. Restart the inference service — it will pick up the models automatically (see `psx_inference/inference.py`)
5. Update `psx_api/predictions/ensemble.py` to call the inference service instead of the heuristic (build-plan step 105)